# Notebook 05: Skill Demand Time-Series Forecasting
## SkillGraph AI - ARIMA, Holt-Winters, Bayesian Prediction Intervals & Emerging Skills

This notebook trains and validates time-series forecasting models to project monthly skill demand shares and identify high-momentum emerging skills.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

conn = sqlite3.connect("../data/skillgraph.db")

# Query forecast results for Python
python_id_row = pd.read_sql_query("SELECT id FROM skills WHERE canonical_name = 'Python'", conn)
if not python_id_row.empty:
    skill_id = int(python_id_row.iloc[0]['id'])
else:
    skill_id = 1

history_df = pd.read_sql_query(f"""
    SELECT period_start, demand_share, posting_count
    FROM skill_demand_observations
    WHERE skill_id = {skill_id}
    ORDER BY period_start ASC
""", conn)

print(f"Retrieved {len(history_df)} historical demand points.")
display(history_df.head())

### 1. Historical Trajectory vs Forecast Projections

In [ ]:
forecast_runs = pd.read_sql_query(f"""
    SELECT id, model_name, mae, rmse, mase
    FROM forecast_runs
    WHERE skill_id = {skill_id}
""", conn)
display(forecast_runs)

if not forecast_runs.empty:
    run_id = int(forecast_runs.iloc[0]['id'])
    forecast_pts = pd.read_sql_query(f"""
        SELECT period_start, forecasted_value, lower_bound, upper_bound
        FROM forecast_results
        WHERE forecast_run_id = {run_id}
        ORDER BY period_start ASC
    """, conn)
    display(forecast_pts)

    plt.figure(figsize=(10, 5))
    if not history_df.empty:
        plt.plot(pd.to_datetime(history_df['period_start']), history_df['demand_share'], 'o-', label="Historical Share", color='#0d9488')
    if not forecast_pts.empty:
        f_dates = pd.to_datetime(forecast_pts['period_start'])
        plt.plot(f_dates, forecast_pts['forecasted_value'], 's--', label="Projected Forecast", color='#0284c7')
        plt.fill_between(f_dates, forecast_pts['lower_bound'], forecast_pts['upper_bound'], color='#0284c7', alpha=0.2, label="95% Prediction Interval")
    plt.title("Skill Demand Trajectory & 95% Confidence Interval")
    plt.xlabel("Period")
    plt.ylabel("Demand Share")
    plt.legend()
    plt.grid(True)
    plt.show()

### 2. Emerging Skills Momentum Classification

In [ ]:
emerging_df = pd.read_sql_query("""
    SELECT s.canonical_name, s.category, AVG(o.demand_share) as current_share
    FROM skill_demand_observations o
    JOIN skills s ON o.skill_id = s.id
    GROUP BY s.id, s.canonical_name, s.category
    ORDER BY current_share DESC
    LIMIT 10
""", conn)
print("Frontier / In-Demand Skills Classification:")
display(emerging_df)
conn.close()